# 06 · Qué aprende un bucle de entrenamiento

Un modelo de una sola recta permite observar el entrenamiento sin esconderlo
detrás de una arquitectura. Cambia el learning rate, el número de épocas o
el tamaño de lote y mira qué cambia en las curvas.

Esta referencia entrena la recta.
`solucion.ipynb` contiene una referencia separada para contrastar tu experimento.
Las piezas de extracción y acumulación son las mismas que la ruta somete a
pruebas. Aquí puedes mirar sus efectos sobre un problema completo.

Datos sintéticos CC0-1.0, generados con semilla fija en el notebook. CPU,
sin descargas. Preparación: entorno de estudio con PyTorch y matplotlib.
60–90 minutos de exploración, sin límite de juez; no es una duración validada con estudiantes.


In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.set_num_threads(1)
g = torch.Generator().manual_seed(23)
def muestras(n, amplitud):
    x = (torch.rand(n, 1, generator=g) * 2 - 1) * amplitud
    y = 2 * x + 1 + 0.06 * torch.randn(n, 1, generator=g)
    return x, y
X_train, y_train = muestras(80, 1.)
X_val, y_val = muestras(40, 1.)
X_nuevo, y_nuevo = muestras(40, 1.5)

plt.scatter(X_train[:, 0], y_train[:, 0], s=15, label="train")
plt.scatter(X_val[:, 0], y_val[:, 0], s=15, label="validación")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()


## Una época y un lote son decisiones distintas

Un microbatch pequeño usa poca memoria. El optimizador puede dar un paso
después de varios microbatches, ponderando cada ejemplo por igual. El último
grupo puede ser más corto: debe producir una actualización igualmente.
Este ejemplo usa MSE media y no contiene dropout ni BatchNorm.


In [ ]:
# bloque: probado; id: acumulacion
def entrenar_con_acumulacion(modelo, cargador, optimizador, perdida,
                             acumular=4, dispositivo="cpu"):
    if not isinstance(acumular, int) or acumular < 1:
        raise ValueError("acumular debe ser un entero positivo")
    modelo.train()  # mover el modelo ANTES de crear el optimizador
    optimizador.zero_grad(set_to_none=True)
    muestras, microbatches, pasos = 0, 0, 0

    def actualizar(n):
        for p in modelo.parameters():
            if p.grad is not None:
                p.grad.div_(n)       # media del lote efectivo, por muestras
        optimizador.step()
        optimizador.zero_grad(set_to_none=True)

    for X, y in cargador:
        if len(X) == 0:
            raise ValueError("un microbatch no puede estar vacío")
        X, y = X.to(dispositivo), y.to(dispositivo)
        (perdida(modelo(X), y) * len(X)).backward()
        muestras += len(X)
        microbatches += 1
        if microbatches == acumular:
            actualizar(muestras)
            muestras, microbatches = 0, 0
            pasos += 1
    if muestras:                    # cola, incluso si hay menos de 4 lotes
        actualizar(muestras)
        pasos += 1
    return pasos


In [ ]:
class EarlyStopping:
    """Corta el entrenamiento cuando la validación deja de mejorar.

    `paciencia` es cuántas épocas sin mejora se toleran antes de parar. En la
    IOAI no es solo una defensa contra el sobreajuste: es una defensa contra el
    reloj, porque el límite de tiempo cubre también la re-ejecución del
    notebook sobre los datos ocultos.
    """

    def __init__(self, paciencia=3, minima_mejora=0.0, modo="min"):
        self.paciencia = paciencia
        self.minima_mejora = minima_mejora
        self.signo = 1.0 if modo == "min" else -1.0
        self.mejor = None
        self.sin_mejorar = 0
        self.mejor_epoca = -1
        self.epoca = -1

    def paso(self, valor):
        """Registra el valor de una época. Devuelve True si hay que parar."""
        self.epoca += 1
        v = self.signo * valor
        if self.mejor is None or v < self.mejor - self.minima_mejora:
            self.mejor = v
            self.mejor_epoca = self.epoca
            self.sin_mejorar = 0
        else:
            self.sin_mejorar += 1
        return self.sin_mejorar >= self.paciencia


## La pérdida elige el checkpoint

Guarda el estado después de registrar la pérdida de esta época. La curva
roja puede dejar de mejorar antes que la azul: observa qué pesos conserva
el entrenamiento. Prueba también un learning rate alto para ver oscilar
la recta. El bloque termina restaurando la mejor validación observada.


In [ ]:
EPOCAS = 80
LR = 0.03
MICROBATCH = 13
ACUMULAR = 4


In [ ]:
torch.manual_seed(7)
modelo = nn.Linear(1, 1)
optimizador = torch.optim.SGD(modelo.parameters(), lr=LR)
cargador = DataLoader(TensorDataset(X_train, y_train), batch_size=MICROBATCH,
                     shuffle=True, generator=torch.Generator().manual_seed(0))
with torch.no_grad():
    baseline = (modelo(X_val) - y_val).abs().mean().item()
parada = EarlyStopping(paciencia=8)
mejor = {k: v.detach().clone() for k, v in modelo.state_dict().items()}
historial, actualizaciones = [], []
for epoca in range(EPOCAS):
    pasos = entrenar_con_acumulacion(modelo, cargador, optimizador, nn.MSELoss(),
                                     acumular=ACUMULAR)
    modelo.eval()
    with torch.no_grad():
        error_train = (modelo(X_train) - y_train).abs().mean().item()
        error_val = (modelo(X_val) - y_val).abs().mean().item()
    historial.append((error_train, error_val))
    actualizaciones.append(pasos)
    detener = parada.paso(error_val)
    if parada.mejor_epoca == parada.epoca:
        mejor = {k: v.detach().clone() for k, v in modelo.state_dict().items()}
    if detener:
        break
modelo.load_state_dict(mejor)
modelo.eval()
with torch.no_grad():
    error_val = (modelo(X_val) - y_val).abs().mean().item()
    error_nuevo = (modelo(X_nuevo) - y_nuevo).abs().mean().item()
resultado = {"baseline": baseline, "validacion": error_val,
             "transferencia": error_nuevo, "mejor_epoca": parada.mejor_epoca,
             "actualizaciones_por_epoca": actualizaciones}

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
if historial:
    curvas = np.asarray(historial)
    ejes[0].plot(curvas[:, 0], label="train")
    ejes[0].plot(curvas[:, 1], label="validación")
    ejes[0].axvline(parada.mejor_epoca, color="black", linestyle="--", label="checkpoint")
ejes[0].axhline(baseline, color="gray", linestyle=":", label="baseline")
ejes[0].set(xlabel="época", ylabel="MAE"); ejes[0].legend()
ejes[1].scatter(X_nuevo[:, 0], y_nuevo[:, 0], s=15, label="transferencia")
rejilla = torch.linspace(-1.5, 1.5, 80).reshape(-1, 1)
with torch.no_grad():
    ejes[1].plot(rejilla[:, 0], modelo(rejilla)[:, 0], color="red", label="modelo")
ejes[1].set(xlabel="x", ylabel="y"); ejes[1].legend()
fig.tight_layout(); plt.show()


## Un error de orden parece un problema del modelo

El encoder de este experimento es la identidad: entrega el propio x.
Un cargador barajado devuelve las filas en otro orden. Al recoger sus
etiquetas juntas, la recta se conserva; al pegar etiquetas de un arreglo
externo, la señal parece desaparecer. En un encoder grande este error es
mucho menos visible, pero tiene la misma causa.


In [ ]:
# bloque: probado; id: extraer_embeddings
import torch

def extraer_embeddings(modelo, cargador, dispositivo="cpu"):
    modelo.to(dispositivo).eval()
    vectores, etiquetas = [], []
    with torch.no_grad():
        for lote, y in cargador:
            # Mover cada resultado a CPU evita acumular toda la tabla en VRAM.
            vectores.append(modelo(lote.to(dispositivo)).detach().cpu())
            etiquetas.append(y.detach().cpu())
    if not vectores:
        raise ValueError("el cargador no puede estar vacío")
    return torch.cat(vectores).numpy(), torch.cat(etiquetas).numpy()


In [ ]:
extractor = nn.Identity()
cargador_features = DataLoader(TensorDataset(X_train, y_train), batch_size=13,
                              shuffle=True, generator=torch.Generator().manual_seed(3))
V, etiquetas = extraer_embeddings(extractor, cargador_features)
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True, sharey=True)
ejes[0].scatter(V[:, 0], etiquetas[:, 0], s=15)
ejes[0].set(title="Features y etiquetas recogidas juntas", xlabel="feature", ylabel="etiqueta")
ejes[1].scatter(V[:, 0], y_train[:, 0], s=15)
ejes[1].set(title="Etiquetas externas: orden equivocado", xlabel="feature")
fig.tight_layout(); plt.show()


## Variantes que revelan el mecanismo

Con 80 ejemplos y microbatches de 13, seis lotes tienen 13 muestras y el
último tiene 2. ¿Cuántas actualizaciones aparecen por época al acumular
cuatro lotes? Prueba luego un microbatch de 80.

La transferencia amplía el intervalo de x sin cambiar la regla generadora.
Imagina ahora una relación curva: una recta puede encajar bien dentro del
intervalo y fallar al extrapolar. El éxito de esta transferencia no valida
cualquier cambio de distribución.


In [ ]:
assert resultado["validacion"] < 0.15 and resultado["transferencia"] < 0.15
assert resultado["validacion"] < resultado["baseline"]
assert all(pasos == 2 for pasos in actualizaciones)
assert abs(modelo.weight.item() - 2) < 0.15
assert abs(modelo.bias.item() - 1) < 0.15
assert np.corrcoef(V[:, 0], etiquetas[:, 0])[0, 1] > 0.98


In [ ]:
resultado.update({"laboratorio": '06_entrenamiento',
                  "version": 'solución de referencia',
                  "metrica": 'MAE en regresión sintética (menor es mejor)', "split": '80 train, 40 validación, 40 transferencia generados con semilla 23; transferencia amplía el intervalo de x'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False,
                  indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
